In [ ]:
# Cell 1: Install dependencies
!pip install uv
!uv pip install --system .

In [ ]:
# Cell 2: Kaggle credentials setup
import os
import json
from pathlib import Path

def load_env_file(path: Path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    os.environ[k.strip()] = v.strip().strip('"').strip("'")

# 1. Try to load from data/kaggle.json if it exists
kaggle_json = Path("./data/kaggle.json")
if kaggle_json.exists():
    try:
        with open(kaggle_json, "r", encoding="utf-8") as f:
            creds = json.load(f)
            os.environ["KAGGLE_USERNAME"] = creds.get("username", "")
            os.environ["KAGGLE_KEY"] = creds.get("key", "")
            print("Kaggle credentials loaded from data/kaggle.json")
    except Exception as e:
        print(f"Error reading data/kaggle.json: {e}")

# 2. Fallback to .env files if not loaded yet
if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
    load_env_file(Path("./.env"))
    load_env_file(Path("../.env"))

# 3. Fallback to Colab Secrets if still not loaded
if "KAGGLE_USERNAME" not in os.environ or "KAGGLE_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        print("Kaggle credentials loaded from Colab Secrets")
    except Exception:
        pass

if "KAGGLE_USERNAME" in os.environ and "KAGGLE_KEY" in os.environ:
    print("Kaggle credentials configured successfully.")
else:
    print("WARNING: Kaggle credentials not found. Please verify ./data/kaggle.json, .env or Colab Secrets.")


In [ ]:
# Cell 3: Download Kaggle arxiv dataset (first run ~3GB, cached after)
import kagglehub, glob

dataset_path = kagglehub.dataset_download("Cornell-University/arxiv")
json_files = glob.glob(f"{dataset_path}/**/*.json", recursive=True)
assert json_files, f"No JSON file found in {dataset_path}"
json_path = json_files[0]
print(f"Kaggle dataset ready: {json_path}")


In [ ]:
# Cell 4: Colab environment setup
import sys, os
from pathlib import Path

def find_backend_root() -> Path:
    """Auto-detect the backend root by looking for known marker directories."""
    candidates = [
        Path("/content/Scientific-research-assistant"),   # git clone default on Colab
        Path(".").resolve(),                   # current working dir
        Path(__file__).resolve().parent.parent if "__file__" in dir() else Path("."),
    ]
    for p in candidates:
        if (p / "dataset_builder").is_dir() and (p / "service").is_dir():
            return p
    raise RuntimeError(
        "Cannot find backend root. "
        "Please set BACKEND_PATH manually or clone to /content/Scientific-research-assistant."
    )

BACKEND_PATH = str(find_backend_root())

if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

os.chdir(BACKEND_PATH)
print(f"Backend root: {BACKEND_PATH}")

# OpenAI API Key
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OpenAI API Key configured from Colab Secrets.")
except Exception:
    if "OPENAI_API_KEY" in os.environ:
        print("OpenAI API Key configured from environment variable.")
    else:
        print("WARNING: OPENAI_API_KEY is not set!")


In [ ]:
# Cell 5: Run the dataset builder using config
import sys
import yaml
sys.path.insert(0, ".") 

from dataset_builder.dataset_builder import DatasetBuilder

# Load configuration from yaml
with open("config/dataset_config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

ds_cfg = cfg["dataset"]
src_cfg = cfg["source"]

builder = DatasetBuilder(
    output_path=ds_cfg["output_path"],
    chunk_size=ds_cfg["chunk_size"],
    overlap=ds_cfg["overlap"],
    samples_per_paper=ds_cfg["samples_per_paper"],
)

builder.build_from_kaggle(
    json_path=json_path,
    n_papers=src_cfg["n_papers"],
    categories=src_cfg["categories"],
    min_year=src_cfg.get("min_year", 2018)
)


In [ ]:
# Cell 6: Inspect results
import json, pandas as pd

samples = [json.loads(l) for l in open("./data/dataset.jsonl", encoding="utf-8")]
print(f"Total samples: {len(samples)}")

df = pd.DataFrame([{
    "paper_id":      s["paper_id"],
    "difficulty":    s["difficulty"],
    "question_type": s["question_type"],
    "section":       s.get("section", ""),
    "answer_len":    len(s["messages"][2]["content"].split()),
} for s in samples])

print("\nDifficulty distribution:")
print(df["difficulty"].value_counts())
print("\nQuestion type distribution:")
print(df["question_type"].value_counts())
df.head(10)


In [ ]:
!git config --global user.email "colab-bot@example.com"
!git config --global user.name "Colab Bot"

try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = os.environ.get("GITHUB_TOKEN", "")

if github_token:
    repo_url = f"https://{github_token}@github.com/XuanNhi183/Scientific-research-assistant.git"

    !git add .
    !git commit -m "chore: auto-update QA dataset from Colab"

    !git push {repo_url} HEAD:main
    print("\nSuccessfully update to GitHub!")
else:
    print("Missing COLAB_TOKEN in environment variables. Cannot push to GitHub.")
